# 01 — RF Baseline LOLO
# Giai đoạn 2 — Mục 2.1 — Random Forest baseline với LOLO
**Đầu ra**: `outputs/tables/rf_baseline_lolo_results.csv`

In [33]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
from common import training

In [34]:
OUTPUT_DIR = Path("./outputs")
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

In [35]:
# Đọc bảng đặc trưng MLP
feature_df = pd.read_parquet("../giai_doan_1_tien_xu_ly/outputs/tables/features_mlp.parquet")
feature_cols = [col for col in feature_df.columns if col.startswith(('time_', 'order_', 'envelope_'))]
X = feature_df[feature_cols].values
y = feature_df['label'].values
loads = feature_df['load_hp'].values

In [36]:
def rf_factory():
    return RandomForestClassifier(n_estimators=100, random_state=42)

per_fold, summary = training.run_lolo_evaluation(
    feature_df,
    feature_cols=feature_cols,
    estimator_factory=rf_factory,
    label_col='label',
    load_col='load_hp',
    val_ratio=0.2,
    seed=42,
    loads=(0, 1, 2, 3),
    use_val_for_fit=True  # Train trên train+val
)

In [37]:
# Mã hóa nhãn
le = LabelEncoder()
y_enc = le.fit_transform(y)

# File-based + LOLO (đọc kết quả đã chạy từ 01)
rf_lolo_per_fold = pd.read_csv(TABLES_DIR / "rf_baseline_lolo_per_fold.csv")
accuracy_lolo = rf_lolo_per_fold['accuracy'].values
f1_lolo = rf_lolo_per_fold['f1_macro'].values
mean_acc_lolo = np.mean(accuracy_lolo)
std_acc_lolo = np.std(accuracy_lolo)
print(f"File-based + LOLO: Accuracy = {mean_acc_lolo:.4f} ± {std_acc_lolo:.4f}")

File-based + LOLO: Accuracy = 1.0000 ± 0.0000


In [38]:
per_fold.to_csv(TABLES_DIR / "rf_baseline_lolo_per_fold.csv", index=False)
summary_df = pd.DataFrame([summary])
summary_df.to_csv(TABLES_DIR / "rf_baseline_lolo_summary.csv", index=False)

print("\n=== Kết quả RF Baseline ===")
print(f"Accuracy: {summary['accuracy_mean']:.4f} ± {summary['accuracy_std']:.4f}")
print(f"F1-macro: {summary['f1_macro_mean']:.4f} ± {summary['f1_macro_std']:.4f}")
per_fold


=== Kết quả RF Baseline ===
Accuracy: 1.0000 ± 0.0000
F1-macro: 1.0000 ± 0.0000


,fold_name,test_load,trainval_loads,n_train,n_val,n_test,accuracy,f1_macro
0,test_load_0,0,"[1, 2, 3]",24,6,10,1.0,1.0
1,test_load_1,1,"[0, 2, 3]",24,6,10,1.0,1.0
2,test_load_2,2,"[0, 1, 3]",24,6,10,1.0,1.0
3,test_load_3,3,"[0, 1, 2]",24,6,10,1.0,1.0


# 2. Random Window Split (chia ngẫu nhiên trên toàn bộ mẫu)

In [39]:
# Lặp 10 lần để lấy trung bình và độ lệch chuẩn
random_accuracies = []
random_f1s = []
for seed in range(10):
    X_train, X_test, y_train, y_test, load_train, load_test = train_test_split(
        X, y_enc, loads, test_size=0.25, random_state=seed, stratify=y_enc
    )
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    random_accuracies.append(acc)
    random_f1s.append(f1)

mean_acc_random = np.mean(random_accuracies)
std_acc_random = np.std(random_accuracies)
mean_f1_random = np.mean(random_f1s)
std_f1_random = np.std(random_f1s)

print(f"Random Window Split: Accuracy = {mean_acc_random:.4f} ± {std_acc_random:.4f}")

Random Window Split: Accuracy = 1.0000 ± 0.0000


# So sánh

In [40]:
delta_acc = mean_acc_random - mean_acc_lolo
print(f"Chênh lệch (Random - LOLO): {delta_acc:.4f}")

# Lưu kết quả bảng
results = pd.DataFrame({
    'Method': ['File-based + LOLO', 'Random Window Split'],
    'Accuracy_mean': [mean_acc_lolo, mean_acc_random],
    'Accuracy_std': [std_acc_lolo, std_acc_random],
    'F1_mean': [np.mean(f1_lolo), mean_f1_random],
    'F1_std': [np.std(f1_lolo), std_f1_random],
    'Delta_Acc': [0, delta_acc]
})
results.to_csv(TABLES_DIR / "rq1_random_vs_lolo.csv", index=False)
print("\nBảng kết quả RQ1:")
print(results)

Chênh lệch (Random - LOLO): 0.0000

Bảng kết quả RQ1:
                Method  Accuracy_mean  Accuracy_std  F1_mean  F1_std  \
0    File-based + LOLO            1.0           0.0      1.0     0.0   
1  Random Window Split            1.0           0.0      1.0     0.0   

   Delta_Acc  
0        0.0  
1        0.0  
